In [7]:
!pip install xgboost


In [15]:
"""
train_adaptpath_v2_final.py
============================================================
FULL END-TO-END TRAINING PIPELINE FOR ADAPTPATH (V2 DATASET)

This script performs:
✔ Data cleaning
✔ Removes unnecessary columns
✔ Removes duplicate interactions
✔ Encoding categorical features
✔ Sequential feature usage
✔ XGBoost model training (optimized)
✔ Accuracy, Precision, Recall, F1, AUC
✔ Confusion Matrix + Classification Report
✔ Saves model + reports
============================================================
"""

import pandas as pd
import numpy as np
import os
import xgboost as xgb
import joblib

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report
)

# ==========================================================
# CONFIG — FIXED DATA PATH
# ==========================================================
# Your dataset is located in: backend/data/
DATA_PATH = "../data/adaptpath_realistic_dataset_v2_10000.csv"

OUT_DIR = "adaptpath_v2_trained_model"
os.makedirs(OUT_DIR, exist_ok=True)

# ==========================================================
# LOAD DATASET
# ==========================================================
print("\n📌 Loading dataset...")

df = pd.read_csv(DATA_PATH)

print("Rows loaded:", len(df))
print(df.head())

# ==========================================================
# REMOVE NON-USEFUL COLUMNS
# ==========================================================
columns_to_drop = ["timestamp", "session_id"]

df = df.drop(columns=[c for c in columns_to_drop if c in df.columns])
print("\nAfter removing unnecessary columns:", df.shape)

# ==========================================================
# REMOVE DUPLICATE STUDENT–QUESTION PAIRS
# ==========================================================
df = df.drop_duplicates(subset=["student_id", "question_id"])
df = df.drop_duplicates()
print("After removing duplicates:", df.shape)

# ==========================================================
# REMOVE LOW-QUALITY ANSWERS
# ==========================================================
df = df[df["time_taken"] >= 5]
df = df[df["time_taken"] <= 150]
df = df.dropna()

print("After cleaning bad rows:", df.shape)

# ==========================================================
# LABEL ENCODING
# ==========================================================
enc_topic = LabelEncoder()
enc_subtopic = LabelEncoder()
enc_diff = LabelEncoder()
enc_style = LabelEncoder()
enc_speed = LabelEncoder()
enc_student = LabelEncoder()

df["topic_enc"] = enc_topic.fit_transform(df["topic"])
df["subtopic_enc"] = enc_subtopic.fit_transform(df["subtopic"])
df["difficulty_enc"] = enc_diff.fit_transform(df["difficulty"])
df["style_enc"] = enc_style.fit_transform(df["learning_style"])
df["speed_enc"] = enc_speed.fit_transform(df["learning_speed"])
df["student_enc"] = enc_student.fit_transform(df["student_id"])

# ==========================================================
# FEATURE MATRIX
# ==========================================================
feature_cols = [
    "ability",
    "time_taken",
    "hints_used",

    "topic_enc",
    "subtopic_enc",
    "difficulty_enc",

    "style_enc",
    "speed_enc",
    "student_enc",

    # Sequential features
    "prev_correct_1",
    "prev_correct_2",
    "prev_correct_3",

    "rolling_accuracy_5",
    "rolling_accuracy_10",

    "topic_attempt_count",
    "topic_correct_rate",
]

X = df[feature_cols]
y = df["correct"]

# ==========================================================
# TRAIN / TEST SPLIT
# ==========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ==========================================================
# TRAIN XGBOOST MODEL
# ==========================================================
print("\n🚀 Training XGBoost model...\n")

model = xgb.XGBClassifier(
    n_estimators=650,
    learning_rate=0.035,
    max_depth=7,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    gamma=0.30,
    eval_metric="logloss",
    nthread=-1,
    random_state=42
)

model.fit(X_train, y_train)

joblib.dump(model, f"{OUT_DIR}/adaptpath_xgb_v2.pkl")
print("✔ Model saved as adaptpath_xgb_v2.pkl")

# ==========================================================
# EVALUATION
# ==========================================================
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

# ==========================================================
# RESULTS
# ==========================================================
print("\n=======================================")
print("   ADAPTPATH MODEL PERFORMANCE (V2)")
print("=======================================\n")

print(classification_report(y_test, y_pred))

print("\n--- Confusion Matrix ---")
print(cm)

print("\n--- Metrics ---")
print(f"Accuracy:      {acc:.4f}")
print(f"Precision:     {prec:.4f}")
print(f"Recall:        {rec:.4f}")
print(f"F1 Score:      {f1:.4f}")
print(f"AUC Score:     {auc:.4f}")

# Save outputs
pd.DataFrame(cm).to_csv(f"{OUT_DIR}/confusion_matrix.csv", index=False)

with open(f"{OUT_DIR}/metrics.txt", "w") as f:
    f.write(f"Accuracy: {acc}\nPrecision: {prec}\nRecall: {rec}\nF1: {f1}\nAUC: {auc}\n")

with open(f"{OUT_DIR}/classification_report.txt", "w") as f:
    f.write(classification_report(y_test, y_pred))

print(f"\n📁 Outputs saved in folder: {OUT_DIR}")
print("\n🎉 TRAINING COMPLETE!")



📌 Loading dataset...
Rows loaded: 10000
  student_id learning_style learning_speed  ability question_id        topic  \
0      S0001         visual       moderate     0.72     Q000094  Probability   
1      S0001         visual       moderate     0.72     Q000035      Algebra   
2      S0001         visual       moderate     0.72     Q000110  Probability   
3      S0001         visual       moderate     0.72     Q000104  Probability   
4      S0001         visual       moderate     0.72     Q000555     Networks   

         subtopic difficulty  time_taken  hints_used  correct  \
0  Expected Value       easy          30           1        0   
1     Polynomials       easy          25           1        0   
2   Distributions       hard          36           1        0   
3  Expected Value     medium          39           0        1   
4             OSI       hard          28           1        0   

             timestamp   session_id  prev_correct_1  prev_correct_2  \
0  2025-02-23T17

In [13]:
print("Full path:", os.path.abspath(DATA_PATH))
print("File exists:", os.path.exists(DATA_PATH))
print("File size:", os.path.getsize(DATA_PATH) if os.path.exists(DATA_PATH) else "NOT FOUND")


Full path: c:\Users\Pooja\Desktop\adaptpath\backend\data\adaptpath_realistic_dataset_v2_10000.csv
File exists: True
File size: 0
